# 03 - Tổng hợp bảng biểu phục vụ báo cáo
Notebook này gom các bảng và chỉ số cuối cùng từ dữ liệu đã làm sạch để chèn trực tiếp vào báo cáo Word.

In [4]:
import os
import pandas as pd

In [5]:
os.makedirs("outputs/report", exist_ok=True)

df = pd.read_csv("data/cleaned_reviews.csv")
summary = pd.read_csv("outputs/eda_summary.csv")

display(df.head())
display(summary)

,review_id,shop_id,item_id,product_name,brand,price,final_price,rating_count_total,user_id,rating_star,...,reply_content,reply_created_at,reply_user_id,reply_is_admin,reply_like_count,source,comment_clean,reply_content_clean,review_len,has_reply
0,Anh KhoiPD DX_201234,FPTShop,59564510814,Colorful Rimbook L1 i5- 13420H/A10205500050,Samsung,16990000.0,14990000.0,1,Anh KhoiPD DX,4.0,...,NaN,NaN,NaN,NaN,NaN,FPTShop,máy dùng tốt,NaN,12,False
1,Võ Diệp Khai_204189,FPTShop,741367556191,MacBook Neo 13 inch 8GB/256GB,Samsung,16490000.0,16490000.0,29,Võ Diệp Khai,5.0,...,NaN,NaN,NaN,NaN,NaN,FPTShop,nhân viên hỗ trợ tue vấn trả góp nhanh chóng,NaN,44,False
2,Chị Nga Xinh Đẹp book hàng_188693,FPTShop,741367556191,MacBook Neo 13 inch 8GB/256GB,Samsung,16490000.0,16490000.0,29,Chị Nga Xinh Đẹp book hàng,5.0,...,NaN,NaN,NaN,NaN,NaN,FPTShop,giá mịm quá ạ,NaN,13,False
3,DƯ_187944,FPTShop,741367556191,MacBook Neo 13 inch 8GB/256GB,Samsung,16490000.0,16490000.0,29,DƯ,5.0,...,NaN,NaN,NaN,NaN,NaN,FPTShop,màu năm nay đẹp phết màu hồng nhìn mê quá mê l...,NaN,57,False
4,Chị Dung_187943,FPTShop,741367556191,MacBook Neo 13 inch 8GB/256GB,Samsung,16490000.0,16490000.0,29,Chị Dung,4.0,...,NaN,NaN,NaN,NaN,NaN,FPTShop,chi phí này thì quá ngon để tậu ngay 1 chiếc m...,NaN,78,False


,metric,value
0,count_reviews,3780.000
1,min_rating,1.000
2,max_rating,5.000
3,avg_rating,4.737
4,median_rating,5.000
5,avg_review_len,108.650
6,min_review_len,2.000
7,max_review_len,424.000


In [6]:
if "has_reply" not in df.columns:
    if "reply_content" in df.columns:
        df["has_reply"] = df["reply_content"].fillna("").astype(str).str.strip().str.len() > 0
    else:
        df["has_reply"] = False

group_cols = ["shop_id", "item_id"]
for optional_col in ["product_name", "brand"]:
    if optional_col in df.columns:
        group_cols.append(optional_col)

table_by_item = df.groupby(group_cols).agg(
    review_count=("review_id", "count"),
    avg_rating=("rating_star", "mean"),
    avg_review_len=("review_len", "mean"),
    reply_count=("has_reply", "sum")
).reset_index()

table_by_item["avg_rating"] = table_by_item["avg_rating"].round(3)
table_by_item["avg_review_len"] = table_by_item["avg_review_len"].round(2)
table_by_item["reply_rate"] = (table_by_item["reply_count"] / table_by_item["review_count"]).round(4)

display(table_by_item.sort_values("review_count", ascending=False).head(20))
table_by_item.to_csv("outputs/report/table_by_item.csv", index=False, encoding="utf-8-sig")

,shop_id,item_id,product_name,brand,review_count,avg_rating,avg_review_len,reply_count,reply_rate
340,FPTShop,543667801489,Asus TUF Gaming FX507ZC4-HN095W i5 12500H,Oppo,118,5.000,130.60,16,0.1356
477,FPTShop,760277157906,Asus Vivobook Flip TN3402YA-LZ188W R5 7530U,Oppo,100,5.000,128.03,45,0.4500
267,FPTShop,431997138342,Asus Vivobook E1404FA-NK186W R5 7520U,Oppo,96,5.000,124.57,39,0.4062
29,FPTShop,28677142814,Asus TUF Gaming FX506HF-HN017W i5 11400H,Oppo,74,5.000,121.69,35,0.4730
234,FPTShop,359513625041,Asus Vivobook X1504VA-NJ526W i5 1335U,Oppo,70,5.000,125.59,22,0.3143
28,FPTShop,28677142627,Asus Vivobook 15 OLED A1505VA-L1113W i5 13500H,Oppo,68,5.000,131.19,19,0.2794
285,FPTShop,461653615435,Asus Vivobook A1405VA-KM257W i5 13500H,Oppo,43,5.000,130.42,4,0.0930
336,FPTShop,543667449885,Asus Vivobook E1504FA-NJ426W R3 7320U,Oppo,41,5.000,128.95,20,0.4878
459,FPTShop,741127624775,Asus Vivobook X1404ZA-NK387W i3 1215U,Oppo,38,4.974,129.11,8,0.2105
401,FPTShop,634869713583,Asus TUF Gaming FA507NU-LP045W R7 7735HS,Oppo,36,5.000,136.25,11,0.3056


In [7]:
table_positive_negative = pd.DataFrame({
    "nhom": ["Tich_cuc_4_5_sao", "Tieu_cuc_1_2_sao", "Trung_tinh_3_sao"],
    "so_luong": [
        int((df["rating_star"] >= 4).sum()),
        int((df["rating_star"] <= 2).sum()),
        int((df["rating_star"] == 3).sum())
    ]
})

display(table_positive_negative)
table_positive_negative.to_csv("outputs/report/table_sentiment_groups.csv", index=False, encoding="utf-8-sig")

,nhom,so_luong
0,Tich_cuc_4_5_sao,3601
1,Tieu_cuc_1_2_sao,96
2,Trung_tinh_3_sao,83


In [8]:
table_data_dictionary = pd.DataFrame([
    ["review_id", "Mã review duy nhất"],
    ["shop_id", "Nguồn bán hoặc mã shop (FPTShop)"],
    ["item_id", "Mã content dùng để gọi API comment của FPTShop"],
    ["product_name", "Tên sản phẩm"],
    ["brand", "Hãng sản phẩm"],
    ["price", "Giá niêm yết hoặc giá tham chiếu"],
    ["final_price", "Giá bán cuối cùng (nếu có)"],
    ["rating_count_total", "Tổng số lượt đánh giá của sản phẩm"],
    ["user_id", "Mã hoặc định danh người dùng đánh giá"],
    ["rating_star", "Số sao đánh giá (1-5)"],
    ["review_title", "Tiêu đề review (nếu có)"],
    ["comment", "Nội dung review gốc"],
    ["comment_clean", "Nội dung review sau làm sạch"],
    ["verified_purchase", "Cờ xác nhận đã mua hàng (nếu có)"],
    ["image_product", "Ảnh đại diện sản phẩm"],
    ["image_review", "Ảnh đính kèm đánh giá (nếu có)"],
    ["created_at", "Thời gian tạo review"],
    ["like_count", "Số lượt hữu ích"],
    ["product_items", "Biến thể/thuộc tính sản phẩm trong review (nếu có)"],
    ["reply_content", "Nội dung phản hồi từ shop/quản trị (nếu có)"],
    ["reply_created_at", "Thời gian phản hồi"],
    ["reply_user_id", "Định danh người phản hồi"],
    ["reply_is_admin", "Cờ phản hồi từ quản trị viên"],
    ["reply_like_count", "Số lượt hữu ích của phản hồi"],
    ["has_reply", "Cờ review có phản hồi từ shop/quản trị"],
    ["review_len", "Độ dài ký tự của review"],
    ["source", "Nguồn dữ liệu (FPTShop)"],
], columns=["ten_cot", "y_nghia"])

display(table_data_dictionary)
table_data_dictionary.to_csv("outputs/report/table_data_dictionary.csv", index=False, encoding="utf-8-sig")

,ten_cot,y_nghia
0,review_id,Mã review duy nhất
1,shop_id,Nguồn bán hoặc mã shop (FPTShop)
2,item_id,Mã content dùng để gọi API comment của FPTShop
3,product_name,Tên sản phẩm
4,brand,Hãng sản phẩm
5,price,Giá niêm yết hoặc giá tham chiếu
6,final_price,Giá bán cuối cùng (nếu có)
7,rating_count_total,Tổng số lượt đánh giá của sản phẩm
8,user_id,Mã hoặc định danh người dùng đánh giá
9,rating_star,Số sao đánh giá (1-5)


In [9]:
print("Đã tạo các bảng trong thư mục outputs/report")

Đã tạo các bảng trong thư mục outputs/report
